# Lecture 1 — RAG Foundations

**Course:** RAG Engineering — From Retrieval to Production

---

## What this notebook covers

| # | Topic |
|---|---|
| 1 | Generative AI fundamentals |
| 2 | LLM limitations |
| 3 | Knowledge cutoff |
| 4 | Hallucination |
| 5 | Context-window limitations |
| 6 | Why RAG? |
| 7 | RAG vs fine-tuning |
| 8 | RAG vs long-context LLMs |
| 9 | RAG architecture |
| 10 | RAG pipeline lifecycle |
| 11 | Indexing pipeline |
| 12 | Retrieval pipeline |
| 13 | Generation pipeline |
| 14 | Grounding and citations |
| 15 | Enterprise use cases |

## How to run it

```bash
pip install numpy pandas matplotlib scikit-learn openai langchain-core langchain-openai
```

That is all you need. **The notebook runs end to end with no API key and no internet.**

Where a real embedding model or LLM would be used, it falls back to a small local
stand-in so the mechanics stay visible. If you *do* have a key, set these before
starting and the same code calls the real thing:

```bash
export EURI_API_KEY="..."
export EURI_CHAT_MODEL="gpt-4.1-mini"
export EURI_EMBEDDING_MODEL="<embedding model enabled on your account>"
pip install openai
```

> **Teaching principle:** do not start with a framework. Start with the problem each piece solves.

In [1]:
!pip install numpy pandas matplotlib scikit-learn openai langchain-core langchain-openai


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Setup — run this first
import os, math, textwrap, re
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# --- optional live backend (EURI). Everything works without it. -------------
EURI_BASE_URL        = "https://api.euron.one/api/v1/euri"
EURI_API_KEY         = os.getenv("EURI_API_KEY")
EURI_CHAT_MODEL      = os.getenv("EURI_CHAT_MODEL", "gpt-4.1-mini")
EURI_EMBEDDING_MODEL = os.getenv("EURI_EMBEDDING_MODEL")
EURI_EMBEDDING_DIMENSION = int(os.getenv("EURI_EMBEDDING_DIMENSION", 1536))

try:
    from openai import OpenAI
    _OPENAI_SDK = True
except ImportError:
    _OPENAI_SDK = False

LIVE_LLM   = bool(EURI_API_KEY and _OPENAI_SDK)
LIVE_EMBED = bool(EURI_API_KEY and EURI_EMBEDDING_MODEL and _OPENAI_SDK)

print("openai sdk installed :", _OPENAI_SDK)
print("live LLM calls       :", LIVE_LLM,   "" if LIVE_LLM   else "-> using a scripted stand-in")
print("live embeddings      :", LIVE_EMBED, "" if LIVE_EMBED else "-> using a local TF-IDF stand-in")
print("embedding dimension  :", EURI_EMBEDDING_DIMENSION)

openai sdk installed : True
live LLM calls       : True 
live embeddings      : True 
embedding dimension  : 1536


## Euri Embedding Model Setup

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EURI_EMBEDDING_MODEL,
    api_key=EURI_API_KEY,
    base_url=EURI_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=EURI_EMBEDDING_DIMENSION
)

print("Embedding model configured:", EURI_EMBEDDING_MODEL)

Embedding model configured: gemini-embedding-2-preview


In [4]:
test_vector = embeddings.embed_query(
    "What is Corrective Retrieval-Augmented Generation?"
)

print("Embedding dimension:", len(test_vector))

Embedding dimension: 1536


In [5]:
test_vector

[-0.0024213625583797693,
 -0.003234507981687784,
 0.008675139397382736,
 0.014268480241298676,
 -0.012205798178911209,
 -0.016287043690681458,
 -0.006864191498607397,
 0.0036184273194521666,
 -0.013394001871347427,
 -0.05471065640449524,
 -0.02056209370493889,
 0.003130154451355338,
 0.009612931869924068,
 -0.0022360417060554028,
 0.01125081442296505,
 -0.011899606324732304,
 0.05366623401641846,
 -0.008887841366231441,
 -0.019573360681533813,
 -0.03550248593091965,
 -0.02695542387664318,
 0.013943742029368877,
 0.023841461166739464,
 0.012781135737895966,
 -0.003394970204681158,
 0.005004988983273506,
 0.019234925508499146,
 -0.013474741019308567,
 0.03270634636282921,
 0.16843293607234955,
 -0.03381215035915375,
 -0.04515211656689644,
 -0.01814260147511959,
 -0.03873325139284134,
 0.013752066530287266,
 0.0285735335201025,
 0.016116896644234657,
 0.030141592025756836,
 0.016381051391363144,
 0.03924043849110603,
 0.024645088240504265,
 -0.009015683084726334,
 -0.016713758930563927,
 

## Euri Chat Model Setup

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=EURI_CHAT_MODEL,
    api_key=EURI_API_KEY,
    base_url=(
        "https://api.euron.one/"
        "api/v1/euri"
    ),
    temperature=0.2,
    max_completion_tokens=1000
)

In [7]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="What is Machine Learning")])
response

AIMessage(content='Machine Learning is a branch of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data, without being explicitly programmed for specific tasks. Instead of following fixed instructions, machine learning systems improve their performance by identifying patterns and relationships in data.\n\nThere are several types of machine learning, including:\n\n1. **Supervised Learning:** The model is trained on labeled data, meaning the input data is paired with the correct output. The goal is to learn a mapping from inputs to outputs to predict outcomes on new, unseen data. Examples include classification and regression tasks.\n\n2. **Unsupervised Learning:** The model is given unlabeled data and must find structure or patterns within it, such as grouping similar data points together (clustering) or reducing data dimensionality.\n\n3. **Reinforcement Learning:** 

In [8]:
response.content

'Machine Learning is a branch of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data, without being explicitly programmed for specific tasks. Instead of following fixed instructions, machine learning systems improve their performance by identifying patterns and relationships in data.\n\nThere are several types of machine learning, including:\n\n1. **Supervised Learning:** The model is trained on labeled data, meaning the input data is paired with the correct output. The goal is to learn a mapping from inputs to outputs to predict outcomes on new, unseen data. Examples include classification and regression tasks.\n\n2. **Unsupervised Learning:** The model is given unlabeled data and must find structure or patterns within it, such as grouping similar data points together (clustering) or reducing data dimensionality.\n\n3. **Reinforcement Learning:** The model learns b

---
# 1. Generative AI fundamentals

An LLM does exactly one thing, over and over: **given the text so far, predict the next token.**

$$P(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

Everything else — answering, summarising, writing code — is that one operation repeated.

Two consequences matter for us:

1. **Fluency is not truth.** The model optimises for *likely* text, not *correct* text. A confident sentence is evidence of nothing.
2. **Knowledge lives in the weights.** Facts are smeared across billions of parameters. There is no row to look up, no source to cite, no record to update.

Let's make next-token prediction concrete with a tiny model we can actually inspect.

In [9]:
# A 2-gram "language model" — the smallest thing that still counts as one.
from collections import defaultdict, Counter

corpus = """
the support team answers refund questions .
the support team answers billing questions .
the support team escalates security questions .
the finance team answers billing questions .
"""

tokens = corpus.split()
bigram = defaultdict(Counter)
for a, b in zip(tokens, tokens[1:]):
    bigram[a][b] += 1

def next_token_distribution(word):
    counts = bigram[word]
    total  = sum(counts.values())
    if total == 0:
        return pd.DataFrame(columns=["next_token", "P"])
    rows = [{"next_token": t, "P": round(c / total, 3)} for t, c in counts.most_common()]
    return pd.DataFrame(rows)

print('After the word "team", the model believes:')
display(next_token_distribution("team"))
print('After the word "answers", the model believes:')
display(next_token_distribution("answers"))

After the word "team", the model believes:


,next_token,P
0,answers,0.75
1,escalates,0.25


After the word "answers", the model believes:


,next_token,P
0,billing,0.667
1,refund,0.333


In [10]:
tokens

['the',
 'support',
 'team',
 'answers',
 'refund',
 'questions',
 '.',
 'the',
 'support',
 'team',
 'answers',
 'billing',
 'questions',
 '.',
 'the',
 'support',
 'team',
 'escalates',
 'security',
 'questions',
 '.',
 'the',
 'finance',
 'team',
 'answers',
 'billing',
 'questions',
 '.']

In [11]:
bigram

defaultdict(collections.Counter,
            {'the': Counter({'support': 3, 'finance': 1}),
             'support': Counter({'team': 3}),
             'team': Counter({'answers': 3, 'escalates': 1}),
             'answers': Counter({'billing': 2, 'refund': 1}),
             'refund': Counter({'questions': 1}),
             'questions': Counter({'.': 4}),
             '.': Counter({'the': 3}),
             'billing': Counter({'questions': 2}),
             'escalates': Counter({'security': 1}),
             'security': Counter({'questions': 1}),
             'finance': Counter({'team': 1})})

In [12]:
bigram["team"]

Counter({'answers': 3, 'escalates': 1})

Notice what just happened. The model has **no idea** what a refund is. It learned that
`team` is usually followed by `answers`. Scale that from 4 sentences to trillions of tokens and
you get something that sounds expert — but the mechanism has not changed.

## Parametric vs external knowledge

| | Parametric knowledge | External knowledge |
|---|---|---|
| Where it lives | inside the model weights | in your documents, databases, APIs |
| When it was fixed | at training time | right now |
| Can you update one fact? | no — retrain | yes — edit the source |
| Can you delete a fact? | not reliably | yes |
| Can you cite it? | no | yes |
| Can you permission it? | no | yes |

**RAG is the bridge between the two columns.** Let's simulate both stores.

In [13]:
# Two knowledge stores: one frozen in the weights, one live in the business.

PARAMETRIC_MEMORY = {                    # what the model "learned" during training
    "cancellation_window_days": 30,      # true in 2024
    "support_hours": "9am-6pm IST",
    "_frozen_at": "2024-06",
}

EXTERNAL_STORE = {                       # what is actually true today, in your systems
    "cancellation_window_days": 14,      # policy changed in 2026
    "support_hours": "24x7",
    "_last_updated": "2026-09-01",
}

comparison = pd.DataFrame(
    [{"fact": k,
      "model believes": PARAMETRIC_MEMORY[k],
      "actually true": EXTERNAL_STORE[k],
      "agrees?": "yes" if PARAMETRIC_MEMORY[k] == EXTERNAL_STORE[k] else "NO"}
     for k in ("cancellation_window_days", "support_hours")]
)
display(comparison)
print("\nThe model is not lying. It is answering correctly — about June 2024.")

,fact,model believes,actually true,agrees?
0,cancellation_window_days,30,14,NO
1,support_hours,9am-6pm IST,24x7,NO



The model is not lying. It is answering correctly — about June 2024.


---
# 2. LLM limitations

Every one of these is a reason RAG exists. Learn them as a checklist: when a RAG system
misbehaves in production, the cause is almost always one of these.

In [14]:
limitations = pd.DataFrame([
 ["Knowledge cutoff", "Facts stop at the training date",
  "Quotes last year's price or policy", "Retrieve the current approved source"],
 ["Hallucination", "Fluent output that nothing supports",
  "Invented refund clause reaches a customer", "Ground generation in supplied evidence"],
 ["No private knowledge", "Your internal docs were never in training",
  "Cannot answer anything company-specific", "Retrieve from your own corpus"],
 ["Context-window limit", "Only N tokens fit in one request",
  "Cannot paste a 40,000-document corpus", "Retrieve a small, relevant subset"],
 ["No provenance", "Parametric answers have no source",
  "Nobody can audit or verify the answer", "Carry metadata and citations through"],
 ["No access control", "The model does not know who may see what",
  "Finance doc leaks into an HR answer", "Filter retrieval by role and tenant"],
 ["Cost and latency", "Every token is paid for, twice over",
  "Long prompts are slow and expensive", "Send the smallest sufficient evidence"],
], columns=["Limitation", "What it means", "What breaks in production", "What RAG does about it"])

display(limitations)
print("Note: RAG does not delete these problems. It gives you a place to fix each one.")

,Limitation,What it means,What breaks in production,What RAG does about it
0,Knowledge cutoff,Facts stop at the training date,Quotes last year's price or policy,Retrieve the current approved source
1,Hallucination,Fluent output that nothing supports,Invented refund clause reaches a customer,Ground generation in supplied evidence
2,No private knowledge,Your internal docs were never in training,Cannot answer anything company-specific,Retrieve from your own corpus
3,Context-window limit,Only N tokens fit in one request,"Cannot paste a 40,000-document corpus","Retrieve a small, relevant subset"
4,No provenance,Parametric answers have no source,Nobody can audit or verify the answer,Carry metadata and citations through
5,No access control,The model does not know who may see what,Finance doc leaks into an HR answer,Filter retrieval by role and tenant
6,Cost and latency,"Every token is paid for, twice over",Long prompts are slow and expensive,Send the smallest sufficient evidence


Note: RAG does not delete these problems. It gives you a place to fix each one.


---
# 3. Knowledge cutoff

> *"What is our current Enterprise Annual cancellation policy?"*

A plain LLM has three options, and all three are bad:

1. answer from stale training data → **confidently wrong**
2. guess a plausible-sounding rule → **hallucination**
3. refuse → **unhelpful**, even though the correct answer exists in your Confluence

Freshness is not one thing. In production it has **four layers**, and any of them can be stale:

| Layer | Question | Owner |
|---|---|---|
| Source freshness | Has the document itself changed? | the business |
| Index freshness | Have we re-ingested it since? | the ingestion pipeline |
| Retrieval freshness | Can the newest version be *found*? | retrieval + filters |
| Answer freshness | Did generation actually *use* it? | context + prompt |

A system can pass layers 1–3 and still fail at 4. Let's watch that happen.

In [15]:
# Closed book vs open book, on a fact that changed.

QUESTION = "How many days do I have to cancel an Enterprise Annual plan?"

def closed_book_answer():
    """A scripted stand-in for what an LLM says from memory alone."""
    d = PARAMETRIC_MEMORY["cancellation_window_days"]
    return f"You can cancel an Enterprise Annual plan within {d} days of purchase."

def open_book_answer(evidence: str):
    """The same question, answered from supplied evidence."""
    m = re.search(r"within (\d+) calendar days", evidence)
    return (f"You can cancel within {m.group(1)} calendar days of purchase. [S1]"
            if m else "The evidence provided does not state a cancellation window.")

CURRENT_POLICY = ("Enterprise Cancellation Policy, version 2026-09. Enterprise Annual plans "
                  "may be cancelled within 14 calendar days of initial purchase.")

print("CLOSED BOOK (model memory):", closed_book_answer())
print("OPEN BOOK  (retrieved)   :", open_book_answer(CURRENT_POLICY))
print("\nSame model. Same question. The difference is entirely the evidence.")

CLOSED BOOK (model memory): You can cancel an Enterprise Annual plan within 30 days of purchase.
OPEN BOOK  (retrieved)   : You can cancel within 14 calendar days of purchase. [S1]

Same model. Same question. The difference is entirely the evidence.


---
# 4. Hallucination

"The model made it up" is not a diagnosis. These five failures look identical to a user and
need completely different fixes — that distinction is the whole job.

In [16]:
hallucination_types = pd.DataFrame([
 ["Intrinsic contradiction", "Answer contradicts the evidence you supplied",
  "Evidence says 14 days; answer says 30", "Generation / prompt"],
 ["Extrinsic fabrication", "Answer adds facts no evidence supports",
  "Invents a 'cooling-off fee'", "Generation / prompt"],
 ["Citation hallucination", "Citation points somewhere that does not support the claim",
  "Cites [S2], but [S2] is about shipping", "Citation validation"],
 ["Retrieval-induced error", "Wrong evidence retrieved, then faithfully used",
  "Retrieved the 2024 policy", "Retrieval / filtering"],
 ["Synthesis error", "Right documents, wrong combination",
  "Merges two plans' rules into one", "Context engineering"],
], columns=["Type", "What it is", "Example", "Where you fix it"])

display(hallucination_types)

,Type,What it is,Example,Where you fix it
0,Intrinsic contradiction,Answer contradicts the evidence you supplied,Evidence says 14 days; answer says 30,Generation / prompt
1,Extrinsic fabrication,Answer adds facts no evidence supports,Invents a 'cooling-off fee',Generation / prompt
2,Citation hallucination,Citation points somewhere that does not support the claim,"Cites [S2], but [S2] is about shipping",Citation validation
3,Retrieval-induced error,"Wrong evidence retrieved, then faithfully used",Retrieved the 2024 policy,Retrieval / filtering
4,Synthesis error,"Right documents, wrong combination",Merges two plans' rules into one,Context engineering


### The grounding rule

Most naive RAG prompts say "answer the question." A production prompt says something closer to:

```text
Use only the supplied evidence for company-specific claims.
If the evidence is insufficient, say so — do not fill the gap.
If sources conflict, state the conflict instead of picking one.
Cite every material claim with its source id, e.g. [S1].
```

Two of those lines matter more than people expect. **"Say so"** gives the model a legal way
out other than inventing. **"State the conflict"** stops it silently choosing the stale version.

In [17]:
GROUNDING_SYSTEM_PROMPT = """You are a grounded enterprise knowledge assistant.

Rules:
1. Use ONLY the evidence below for company-specific claims.
2. If the evidence is insufficient, say exactly: "The available sources do not answer this."
3. If sources conflict, state the conflict and cite both.
4. Cite every material factual claim with its source id, e.g. [S1].
5. Do not add information from your own training data."""

def build_prompt(question: str, evidence_blocks: List[str]) -> str:
    evidence = "\n\n".join(evidence_blocks)
    return f"""{GROUNDING_SYSTEM_PROMPT}

EVIDENCE
{evidence}

QUESTION
{question}

ANSWER (with citations):"""

demo = build_prompt(
    "What is the cancellation window for the Enterprise Annual plan?",
    ["[S1] Enterprise Cancellation Policy v2026-09: Enterprise Annual plans may be "
     "cancelled within 14 calendar days of initial purchase.",
     "[S2] Enterprise Provisioning Rules: cancellation eligibility ends once "
     "provisioning has completed."]
)
print(demo)

You are a grounded enterprise knowledge assistant.

Rules:
1. Use ONLY the evidence below for company-specific claims.
2. If the evidence is insufficient, say exactly: "The available sources do not answer this."
3. If sources conflict, state the conflict and cite both.
4. Cite every material factual claim with its source id, e.g. [S1].
5. Do not add information from your own training data.

EVIDENCE
[S1] Enterprise Cancellation Policy v2026-09: Enterprise Annual plans may be cancelled within 14 calendar days of initial purchase.

[S2] Enterprise Provisioning Rules: cancellation eligibility ends once provisioning has completed.

QUESTION
What is the cancellation window for the Enterprise Annual plan?

ANSWER (with citations):


---
# 5. Context-window limitations

Modern models take very large contexts. So why not paste the entire company?

Because the context window is a **desk, not a warehouse**. Six reasons, roughly in order of
how often they bite:

| Reason | What it costs you |
|---|---|
| Money | You pay per input token, on every single request |
| Latency | A bigger prompt is a slower response |
| Signal dilution | The right paragraph gets buried — "lost in the middle" |
| Version conflict | The 2024 and 2026 policies both arrive, and the model picks one |
| Security | Everything in the prompt is visible to the model, permissions or not |
| Debuggability | When the answer is wrong, which of 900 pages caused it? |

Let's put numbers on the first two.

In [18]:
# Rough but useful: ~4 characters per token for English prose.
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

# A small enterprise corpus: 8,000 documents averaging 2 pages each.
N_DOCS          = 8_000
TOKENS_PER_DOC  = 1_200
PRICE_PER_1M_IN = 0.15        # USD per 1M input tokens — set to your own model's price
QUERIES_PER_DAY = 5_000

stuff_everything = N_DOCS * TOKENS_PER_DOC
retrieve_five    = 5 * 400 + 300                     # 5 chunks + question and instructions

rows = []
for name, tokens in [("Paste the whole corpus", stuff_everything),
                     ("Retrieve 5 chunks", retrieve_five)]:
    cost_per_query = tokens / 1_000_000 * PRICE_PER_1M_IN
    rows.append({
        "approach": name,
        "input tokens / query": f"{tokens:,}",
        "cost / query (USD)": f"${cost_per_query:,.4f}",
        "cost / day (USD)": f"${cost_per_query * QUERIES_PER_DAY:,.2f}",
        "cost / year (USD)": f"${cost_per_query * QUERIES_PER_DAY * 365:,.0f}",
    })

display(pd.DataFrame(rows))
print(f"Ratio: retrieval sends {stuff_everything / retrieve_five:,.0f}x fewer tokens per query.")
print("And that is before latency, dilution, conflicts, security and debuggability.")

,approach,input tokens / query,cost / query (USD),cost / day (USD),cost / year (USD)
0,Paste the whole corpus,"9,600,000",$1.4400,"$7,200.00","$2,628,000"
1,Retrieve 5 chunks,"2,300",$0.0003,$1.72,$630


Ratio: retrieval sends 4,174x fewer tokens per query.
And that is before latency, dilution, conflicts, security and debuggability.


In [19]:
((8_000*1_200)/1_000_000)*0.15*5_000


7200.0

---
# 6. Why RAG?

RAG is an **inference-time architecture**. Nothing is retrained. At the moment a question arrives:

1. take the query
2. search an external knowledge store
3. select the evidence that matters
4. build a context out of it
5. ask the LLM to answer **using that evidence**
6. return the answer with its sources

Formally, with $q$ the query, $K$ the knowledge index, $I$ the instructions:

$$D_q = \text{Retrieve}(q, K) \qquad a = \text{LLM}(q, D_q, I)$$

The whole of the rest of this course is making `Retrieve` better and `LLM(...)` more disciplined.

Here is the entire idea in about twenty lines — no framework, no vector database yet.

In [20]:
# RAG in twenty lines, so you can see there is no magic in it.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mini_docs = [
    "Enterprise Annual plans may be cancelled within 14 calendar days of purchase.",
    "Support is available 24x7 for Enterprise customers via chat and phone.",
    "Invoices are issued on the first business day of each month.",
]

_vec  = TfidfVectorizer().fit(mini_docs)
_dmat = _vec.transform(mini_docs)

def mini_rag(question, k=1):
    qv     = _vec.transform([question])                  # 1. embed the query
    scores = cosine_similarity(qv, _dmat)[0]             # 2. compare to the index
    top    = np.argsort(scores)[::-1][:k]                # 3. select evidence
    ctx    = "\n".join(f"[S{i+1}] {mini_docs[j]}" for i, j in enumerate(top))
    return ctx, build_prompt(question, [ctx])            # 4. build the context

context, prompt = mini_rag("how long do I have to cancel?")
print("RETRIEVED EVIDENCE:\n" + context)
print("\nThat context is the entire difference between a guess and an answer.")

RETRIEVED EVIDENCE:
[S1] Invoices are issued on the first business day of each month.

That context is the entire difference between a guess and an answer.


In [21]:
_vec

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [22]:
# how to convert sparse matrix to dense matrix
_dmat.toarray()

array([[0.29938511, 0.        , 0.        , 0.29938511, 0.        ,
        0.        , 0.29938511, 0.        , 0.29938511, 0.29938511,
        0.        , 0.        , 0.        , 0.29938511, 0.        ,
        0.22769009, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.29938511, 0.        , 0.22769009, 0.        ,
        0.        , 0.29938511, 0.29938511, 0.        , 0.        ,
        0.        , 0.29938511],
       [0.        , 0.30746099, 0.30746099, 0.        , 0.        ,
        0.30746099, 0.        , 0.        , 0.        , 0.        ,
        0.30746099, 0.30746099, 0.        , 0.        , 0.        ,
        0.23383201, 0.        , 0.30746099, 0.        , 0.30746099,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.30746099, 0.        , 0.        , 0.30746099, 0.        ,
        0.30746099, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.30746099,
        0.        , 0.        , 0.30746099, 0.    

---
# 7. RAG vs fine-tuning

The single most common interview question on this topic, and the single most common
architecture mistake. They solve **different problems**.

> **The one-line rule:** RAG changes what the model *knows*. Fine-tuning changes how the model *behaves*.

In [23]:
versus_ft = pd.DataFrame([
 ["Facts that change weekly",          "RAG",          "Edit the source, re-index. Done."],
 ["Private company documents",         "RAG",          "Retrieve them; no training run needed."],
 ["Citations and audit trail",         "RAG",          "Retrieval knows where evidence came from."],
 ["Deleting a fact (GDPR, expiry)",    "RAG",          "Delete the row. Unlearning weights is unsolved."],
 ["A large, growing corpus",           "RAG",          "Weights are not a database."],
 ["Always answer in our house format", "Fine-tuning",  "That is behaviour, not knowledge."],
 ["Domain tone, style, terminology",   "Fine-tuning",  "Teach the pattern once."],
 ["Structured output the model keeps breaking", "Fine-tuning", "Train the shape of the output."],
 ["Lower latency on a narrow task",    "Fine-tuning",  "A smaller tuned model can beat a big prompt."],
], columns=["Your requirement", "Choose", "Why"])

display(versus_ft)

def choose_approach(requirement: str) -> str:
    """A deliberately crude router — the point is the reasoning, not the regex."""
    r = requirement.lower()
    knowledge = ["fact", "document", "policy", "current", "latest", "cite",
                 "source", "private", "update", "delete", "corpus"]
    behaviour = ["tone", "style", "format", "persona", "structure",
                 "always respond", "terminology", "shorter", "latency"]
    if any(w in r for w in knowledge):
        return "RAG (knowledge problem)"
    if any(w in r for w in behaviour):
        return "Fine-tuning (behaviour problem)"
    return "Unclear — ask whether the gap is knowledge or behaviour"

for req in ["We need the latest HR policy answered correctly",
            "Every reply must follow our legal tone and format",
            "Support agents need answers from private runbooks",
            "The model must always emit valid JSON in our schema"]:
    print(f"{choose_approach(req):<42} <- {req}")

,Your requirement,Choose,Why
0,Facts that change weekly,RAG,"Edit the source, re-index. Done."
1,Private company documents,RAG,Retrieve them; no training run needed.
2,Citations and audit trail,RAG,Retrieval knows where evidence came from.
3,"Deleting a fact (GDPR, expiry)",RAG,Delete the row. Unlearning weights is unsolved.
4,"A large, growing corpus",RAG,Weights are not a database.
5,Always answer in our house format,Fine-tuning,"That is behaviour, not knowledge."
6,"Domain tone, style, terminology",Fine-tuning,Teach the pattern once.
7,Structured output the model keeps breaking,Fine-tuning,Train the shape of the output.
8,Lower latency on a narrow task,Fine-tuning,A smaller tuned model can beat a big prompt.


RAG (knowledge problem)                    <- We need the latest HR policy answered correctly
Fine-tuning (behaviour problem)            <- Every reply must follow our legal tone and format
RAG (knowledge problem)                    <- Support agents need answers from private runbooks
Unclear — ask whether the gap is knowledge or behaviour <- The model must always emit valid JSON in our schema


---
# 8. RAG vs long-context LLMs

"Context windows are huge now — is RAG obsolete?" No, and the reason is not about size.

Long context is genuinely better when the corpus is **small and you need all of it** —
analysing one contract end to end, summarising a single long report. Below roughly a few
hundred pages that you own entirely, just paste it.

RAG wins the moment any of these is true, and in an enterprise at least one always is:

In [24]:
versus_lc = pd.DataFrame([
 ["Corpus size",        "Fits in the window",         "Millions of chunks",            "RAG"],
 ["Cost per query",     "Pay for everything, always", "Pay for ~5 chunks",             "RAG"],
 ["Latency",            "Grows with the prompt",      "Roughly flat",                  "RAG"],
 ["Access control",     "All-or-nothing",             "Filter per user, per tenant",   "RAG"],
 ["Conflicting versions","Both versions go in",       "Filter to the current one",     "RAG"],
 ["Why this answer?",   "Somewhere in 900 pages",     "These 5 chunks, by id",         "RAG"],
 ["One-off deep read",  "Simple and effective",       "Index-then-query is overhead",  "Long context"],
 ["Whole-document reasoning", "Sees everything at once", "May miss cross-references",  "Long context"],
], columns=["Dimension", "Long context", "RAG", "Winner"])

display(versus_lc)
print("The mature pattern is not either/or:")
print("  retrieval selects the evidence  ->  a generous context window reasons over it.")

,Dimension,Long context,RAG,Winner
0,Corpus size,Fits in the window,Millions of chunks,RAG
1,Cost per query,"Pay for everything, always",Pay for ~5 chunks,RAG
2,Latency,Grows with the prompt,Roughly flat,RAG
3,Access control,All-or-nothing,"Filter per user, per tenant",RAG
4,Conflicting versions,Both versions go in,Filter to the current one,RAG
5,Why this answer?,Somewhere in 900 pages,"These 5 chunks, by id",RAG
6,One-off deep read,Simple and effective,Index-then-query is overhead,Long context
7,Whole-document reasoning,Sees everything at once,May miss cross-references,Long context


The mature pattern is not either/or:
  retrieval selects the evidence  ->  a generous context window reasons over it.


---
# 9. RAG architecture

A RAG system is not one pipeline. It is **three planes** that run on different clocks — and
confusing them is why teams debug the wrong thing.

In [25]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_lane(ax, y, title, steps, colour):
    ax.text(0.02, y + 0.135, title, fontsize=11.5, weight="bold", color=colour,
            transform=ax.transAxes)
    n  = len(steps)
    xs = np.linspace(0.02, 0.86, n)
    w  = 0.84 / n * 0.86
    for i, (x, label) in enumerate(zip(xs, steps)):
        ax.add_patch(FancyBboxPatch((x, y), w, 0.10, boxstyle="round,pad=0.004,rounding_size=0.01",
                                    fc="white", ec=colour, lw=1.6, transform=ax.transAxes))
        ax.text(x + w / 2, y + 0.05, label, ha="center", va="center", fontsize=8.6,
                transform=ax.transAxes)
        if i < n - 1:
            ax.add_patch(FancyArrowPatch((x + w, y + 0.05), (xs[i + 1], y + 0.05),
                                         arrowstyle="-|>", mutation_scale=11, color="#9AA3C7",
                                         lw=1.4, transform=ax.transAxes))

fig, ax = plt.subplots(figsize=(13, 5.2))
ax.axis("off")
draw_lane(ax, 0.74, "A · INDEXING PLANE   (offline, runs on a schedule)",
          ["Sources", "Parse", "Clean", "Chunk", "Metadata", "Embed", "Index"], "#1F9FD4")
draw_lane(ax, 0.42, "B · SERVING PLANE   (online, runs per question)",
          ["Question", "Auth / ACL", "Retrieve", "Rerank", "Context", "Generate", "Cite"], "#12A882")
draw_lane(ax, 0.10, "C · OPERATIONS PLANE   (continuous)",
          ["Evaluate", "Trace", "Secure", "Latency", "Cost", "Feedback"], "#E8A317")
plt.tight_layout(); plt.savefig("rag_planes.png", dpi=150, bbox_inches="tight")

print("Plane A is a data pipeline.  Plane B is a request path.  Plane C is how you keep both honest.")

Plane A is a data pipeline.  Plane B is a request path.  Plane C is how you keep both honest.


Why the split matters when something goes wrong:

- the answer is **stale** → plane A (did we re-ingest?)
- the answer is **wrong but confident** → plane B (what did we retrieve, what did we send?)
- nobody **noticed** it was wrong → plane C (no evaluation, no tracing)

---
# 10. RAG pipeline lifecycle

Sixteen stages. You do not build all of them in week 1 — but you should know which stage
you are standing in when something breaks.

In [26]:
lifecycle = pd.DataFrame([
 [1,  "Data acquisition", "A", "Connect to the source systems",        "Source missed entirely"],
 [2,  "Ingestion",        "A", "Pull content and detect changes",      "Stale index"],
 [3,  "Parsing",          "A", "Text, tables, layout, metadata",       "Table turns into noise"],
 [4,  "Normalisation",    "A", "Strip boilerplate, fix encoding",      "Headers repeated in every chunk"],
 [5,  "Chunking",         "A", "Create the unit of retrieval",         "Rule split from its condition"],
 [6,  "Embedding",        "A", "Text to vectors",                      "Wrong model for the domain"],
 [7,  "Indexing",         "A", "Persist vectors + payload + ids",      "No metadata to filter on"],
 [8,  "Query understanding","B","Normalise, classify, rewrite",        "Pronouns with no context"],
 [9,  "Retrieval",        "B", "Dense, sparse, hybrid, graph, SQL",    "Right doc never surfaces"],
 [10, "Reranking",        "B", "Reorder with a stronger signal",       "Best chunk sits at rank 20"],
 [11, "Context engineering","B","Select, order, compress, label",      "Lost in the middle"],
 [12, "Generation",       "B", "Produce the grounded answer",          "Ignores the evidence"],
 [13, "Citation",         "B", "Map claims back to sources",           "Citation does not support claim"],
 [14, "Evaluation",       "C", "Measure retrieval and generation",     "Regressions ship silently"],
 [15, "Observability",    "C", "Trace latency, tokens, decisions",     "Cannot reproduce a bad answer"],
 [16, "Feedback loop",    "C", "Improve corpus, retrieval, prompts",   "Same failure every week"],
], columns=["#", "Stage", "Plane", "What it does", "Classic failure"])

display(lifecycle)

,#,Stage,Plane,What it does,Classic failure
0,1,Data acquisition,A,Connect to the source systems,Source missed entirely
1,2,Ingestion,A,Pull content and detect changes,Stale index
2,3,Parsing,A,"Text, tables, layout, metadata",Table turns into noise
3,4,Normalisation,A,"Strip boilerplate, fix encoding",Headers repeated in every chunk
4,5,Chunking,A,Create the unit of retrieval,Rule split from its condition
5,6,Embedding,A,Text to vectors,Wrong model for the domain
6,7,Indexing,A,Persist vectors + payload + ids,No metadata to filter on
7,8,Query understanding,B,"Normalise, classify, rewrite",Pronouns with no context
8,9,Retrieval,B,"Dense, sparse, hybrid, graph, SQL",Right doc never surfaces
9,10,Reranking,B,Reorder with a stronger signal,Best chunk sits at rank 20


---
# 11. Indexing pipeline

Everything that happens **before** anyone asks a question.

The one idea to take away: **a chunk is the unit of retrieval.** You will never retrieve half
a chunk, and you will never retrieve across two chunks. So a chunk must be able to stand alone
and still make sense.

Ask of every chunk:

- does it contain a complete idea?
- does it carry the heading it needs to be interpreted?
- is a rule separated from its condition?
- is a table separated from its caption?
- does it carry the metadata needed to filter and cite it?

In [27]:
# A small corpus with the things that make real corpora hard:
#   - two versions of the same policy (one superseded)
#   - department-restricted documents
#   - deliberately irrelevant content

CORPUS = [
 dict(doc_id="POL-CANCEL-2024", title="Enterprise Cancellation Policy",
      version="2024-03", status="superseded", department="legal", access_roles=["all"],
      text=("Enterprise Annual plans may be cancelled within 30 calendar days of initial "
            "purchase. A cancellation request must be submitted by an account administrator. "
            "Refunds are processed within two billing cycles.")),
 dict(doc_id="POL-CANCEL-2026", title="Enterprise Cancellation Policy",
      version="2026-09", status="active", department="legal", access_roles=["all"],
      text=("Enterprise Annual plans may be cancelled within 14 calendar days of initial "
            "purchase. A cancellation request must be submitted by an account administrator. "
            "Refunds are processed within one billing cycle. This version supersedes the 2024 policy.")),
 dict(doc_id="POL-PROVISION", title="Enterprise Provisioning Rules",
      version="2026-04", status="active", department="legal", access_roles=["all"],
      text=("Cancellation eligibility ends once enterprise provisioning has been completed, "
            "even if the calendar window has not expired.")),
 dict(doc_id="SUP-HOURS", title="Support Availability",
      version="2026-01", status="active", department="support", access_roles=["all"],
      text=("Enterprise customers receive 24x7 support through chat and telephone. "
            "Standard customers receive support between 9am and 6pm IST on business days.")),
 dict(doc_id="FIN-SALARY", title="Compensation Bands FY26",
      version="2026-04", status="active", department="finance", access_roles=["finance"],
      text=("Compensation bands for FY26 are reviewed each April by the finance committee. "
            "Band movement requires director approval.")),
 dict(doc_id="ENG-RUNBOOK", title="Search Latency Runbook",
      version="2026-07", status="active", department="engineering", access_roles=["engineering"],
      text=("If p95 search latency exceeds 800ms, check the vector index replica count first, "
            "then embedding service saturation, then the reranker queue depth.")),
 dict(doc_id="MKT-BLOG", title="Why We Rebranded",
      version="2025-11", status="active", department="marketing", access_roles=["all"],
      text=("Our rebrand focused on clarity and a warmer colour palette. The logo was "
            "redrawn to work at small sizes.")),
]

print(f"{len(CORPUS)} documents loaded.")
display(pd.DataFrame(CORPUS)[["doc_id", "title", "version", "status", "department", "access_roles"]])

7 documents loaded.


,doc_id,title,version,status,department,access_roles
0,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all]
1,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all]
2,POL-PROVISION,Enterprise Provisioning Rules,2026-04,active,legal,[all]
3,SUP-HOURS,Support Availability,2026-01,active,support,[all]
4,FIN-SALARY,Compensation Bands FY26,2026-04,active,finance,[finance]
5,ENG-RUNBOOK,Search Latency Runbook,2026-07,active,engineering,[engineering]
6,MKT-BLOG,Why We Rebranded,2025-11,active,marketing,[all]


In [28]:
# Chunking with overlap, carrying metadata down to every chunk.

def chunk_text(text: str, size: int = 40, overlap: int = 8) -> List[str]:
    """Word-window chunking. Deliberately simple — week 3 replaces this properly."""
    if size <= overlap:
        raise ValueError("size must be greater than overlap, or chunking never advances")
    words, out, start = text.split(), [], 0
    while start < len(words):
        end = min(start + size, len(words))
        out.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return out

def build_chunks(corpus: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks = []
    for doc in corpus:
        for i, piece in enumerate(chunk_text(doc["text"])):
            chunks.append({
                "chunk_id":     f'{doc["doc_id"]}#{i}',
                "doc_id":       doc["doc_id"],
                "title":        doc["title"],
                "version":      doc["version"],
                "status":       doc["status"],
                "department":   doc["department"],
                "access_roles": doc["access_roles"],
                # the heading is prepended so the chunk can stand alone when retrieved
                "text":         f'{doc["title"]} ({doc["version"]}): {piece}',
            })
    return chunks

CHUNKS = build_chunks(CORPUS)
print(f"{len(CORPUS)} documents -> {len(CHUNKS)} chunks")
display(pd.DataFrame(CHUNKS)[["chunk_id", "status", "department", "text"]].head(10))

7 documents -> 7 chunks


,chunk_id,status,department,text
0,POL-CANCEL-2024#0,superseded,legal,Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...
1,POL-CANCEL-2026#0,active,legal,Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...
2,POL-PROVISION#0,active,legal,Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...
3,SUP-HOURS#0,active,support,Support Availability (2026-01): Enterprise customers receive 24x7 support through chat...
4,FIN-SALARY#0,active,finance,Compensation Bands FY26 (2026-04): Compensation bands for FY26 are reviewed each April...
5,ENG-RUNBOOK#0,active,engineering,"Search Latency Runbook (2026-07): If p95 search latency exceeds 800ms, check the vecto..."
6,MKT-BLOG#0,active,marketing,Why We Rebranded (2025-11): Our rebrand focused on clarity and a warmer colour palette...


In [29]:
CHUNKS

[{'chunk_id': 'POL-CANCEL-2024#0',
  'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled within 30 calendar days of initial purchase. A cancellation request must be submitted by an account administrator. Refunds are processed within two billing cycles.'},
 {'chunk_id': 'POL-CANCEL-2026#0',
  'doc_id': 'POL-CANCEL-2026',
  'title': 'Enterprise Cancellation Policy',
  'version': '2026-09',
  'status': 'active',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled within 14 calendar days of initial purchase. A cancellation request must be submitted by an account administrator. Refunds are processed within one billing cycle. This version supersedes the 2024 policy.'},
 {'chunk_id': 'P

In [30]:
# Chunking with overlap, carrying metadata down to every chunk.

def chunk_text(text: str, size: int = 20, overlap: int = 4) -> List[str]:
    """Word-window chunking. Deliberately simple — week 3 replaces this properly."""
    if size <= overlap:
        raise ValueError("size must be greater than overlap, or chunking never advances")
    words, out, start = text.split(), [], 0
    while start < len(words):
        end = min(start + size, len(words))
        out.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return out

def build_chunks(corpus: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks = []
    for doc in corpus:
        for i, piece in enumerate(chunk_text(doc["text"])):
            chunks.append({
                "chunk_id":     f'{doc["doc_id"]}#{i}',
                "doc_id":       doc["doc_id"],
                "title":        doc["title"],
                "version":      doc["version"],
                "status":       doc["status"],
                "department":   doc["department"],
                "access_roles": doc["access_roles"],
                # the heading is prepended so the chunk can stand alone when retrieved
                "text":         f'{doc["title"]} ({doc["version"]}): {piece}',
            })
    return chunks

CHUNKS = build_chunks(CORPUS)
print(f"{len(CORPUS)} documents -> {len(CHUNKS)} chunks")
display(pd.DataFrame(CHUNKS)[["chunk_id", "status", "department", "text"]].head(10))

7 documents -> 11 chunks


,chunk_id,status,department,text
0,POL-CANCEL-2024#0,superseded,legal,Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...
1,POL-CANCEL-2024#1,superseded,legal,Enterprise Cancellation Policy (2024-03): must be submitted by an account administrato...
2,POL-CANCEL-2026#0,active,legal,Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...
3,POL-CANCEL-2026#1,active,legal,Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...
4,POL-PROVISION#0,active,legal,Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...
5,SUP-HOURS#0,active,support,Support Availability (2026-01): Enterprise customers receive 24x7 support through chat...
6,SUP-HOURS#1,active,support,Support Availability (2026-01): 6pm IST on business days.
7,FIN-SALARY#0,active,finance,Compensation Bands FY26 (2026-04): Compensation bands for FY26 are reviewed each April...
8,ENG-RUNBOOK#0,active,engineering,"Search Latency Runbook (2026-07): If p95 search latency exceeds 800ms, check the vecto..."
9,ENG-RUNBOOK#1,active,engineering,"Search Latency Runbook (2026-07): saturation, then the reranker queue depth."


In [31]:
CHUNKS

[{'chunk_id': 'POL-CANCEL-2024#0',
  'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled within 30 calendar days of initial purchase. A cancellation request must be submitted by'},
 {'chunk_id': 'POL-CANCEL-2024#1',
  'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2024-03): must be submitted by an account administrator. Refunds are processed within two billing cycles.'},
 {'chunk_id': 'POL-CANCEL-2026#0',
  'doc_id': 'POL-CANCEL-2026',
  'title': 'Enterprise Cancellation Policy',
  'version': '2026-09',
  'status': 'active',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cance

### Embedding and indexing

In production this is an embedding model (through EURI, OpenAI, Cohere, a local model)
writing into a vector database (Qdrant, Pinecone, Weaviate, Milvus, pgvector).

For this lecture we use a **local TF-IDF stand-in** so the notebook runs anywhere. It is
genuinely lexical, not semantic — which is a useful thing for students to *see*, because it
fails in exactly the way week 5's hybrid-search lecture is about. If `EURI_EMBEDDING_MODEL`
is set, the same function calls the real model instead.

In [32]:
class LocalEmbedder:
    """Offline stand-in for an embedding model. TF-IDF vectors, L2-normalised."""
    name = "local-tfidf (offline stand-in)"

    def __init__(self, texts: List[str]):
        self.vec = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit(texts)

    def encode(self, texts: List[str]) -> np.ndarray:
        return np.asarray(self.vec.transform(texts).todense())

class EuriEmbedder:
    """The real thing, through the EURI gateway."""
    def __init__(self, model: str):
        self.model  = model
        self.name   = f"euri:{model}"
        self.dimensions = EURI_EMBEDDING_DIMENSION
        self.client = OpenAI(api_key=EURI_API_KEY, base_url=EURI_BASE_URL)

    def encode(self, texts: List[str]) -> np.ndarray:
        r = self.client.embeddings.create(model=self.model, input=texts)
        return np.asarray([d.embedding for d in r.data], dtype=float)

chunk_texts = [c["text"] for c in CHUNKS]
embedder    = EuriEmbedder(EURI_EMBEDDING_MODEL) if LIVE_EMBED else LocalEmbedder(chunk_texts)
INDEX       = embedder.encode(chunk_texts)

print("embedder     :", embedder.name)
print("index shape  :", INDEX.shape, "-> (chunks, dimensions)")

embedder     : euri:gemini-embedding-2-preview
index shape  : (11, 3072) -> (chunks, dimensions)


In [33]:
CHUNKS

[{'chunk_id': 'POL-CANCEL-2024#0',
  'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled within 30 calendar days of initial purchase. A cancellation request must be submitted by'},
 {'chunk_id': 'POL-CANCEL-2024#1',
  'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cancellation Policy (2024-03): must be submitted by an account administrator. Refunds are processed within two billing cycles.'},
 {'chunk_id': 'POL-CANCEL-2026#0',
  'doc_id': 'POL-CANCEL-2026',
  'title': 'Enterprise Cancellation Policy',
  'version': '2026-09',
  'status': 'active',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Cance

In [34]:
chunk_texts

['Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled within 30 calendar days of initial purchase. A cancellation request must be submitted by',
 'Enterprise Cancellation Policy (2024-03): must be submitted by an account administrator. Refunds are processed within two billing cycles.',
 'Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled within 14 calendar days of initial purchase. A cancellation request must be submitted by',
 'Enterprise Cancellation Policy (2026-09): must be submitted by an account administrator. Refunds are processed within one billing cycle. This version supersedes the 2024 policy.',
 'Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise provisioning has been completed, even if the calendar window has not expired.',
 'Support Availability (2026-01): Enterprise customers receive 24x7 support through chat and telephone. Standard customers receive support between 9am and 

In [35]:
INDEX

array([[-0.00411409,  0.02353633, -0.00122807, ...,  0.00632847,
        -0.00904615,  0.01346349],
       [ 0.00381948,  0.01146675,  0.02007654, ..., -0.00636809,
        -0.00613326,  0.00078324],
       [-0.00571726,  0.02212957, -0.00795158, ...,  0.00041128,
        -0.01240145,  0.02055717],
       ...,
       [ 0.01235981,  0.00445156,  0.00121917, ...,  0.00382775,
        -0.00117398, -0.01482094],
       [ 0.0039745 ,  0.00985981, -0.00257155, ...,  0.00180747,
         0.01052988, -0.00980691],
       [ 0.00956105,  0.01853756, -0.00449431, ..., -0.01285615,
         0.01710619,  0.00536923]], shape=(11, 3072))

---
# 12. Retrieval pipeline

Given a question, find the smallest set of chunks that could answer it.

The similarity itself is one line of maths — the **cosine** of the angle between two vectors:

$$\cos(\theta) = \frac{\mathbf{q} \cdot \mathbf{d}}{\lVert \mathbf{q} \rVert \, \lVert \mathbf{d} \rVert}$$

Close to 1 means pointing the same way; close to 0 means unrelated. That is the whole formula.

Everything that makes retrieval *good* is the machinery around it: filters, freshness,
permissions, reranking. Let's build those one at a time and watch the answer change.

In [36]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a, float).ravel(), np.asarray(b, float).ravel()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

# sanity check on vectors you can verify by eye
q  = np.array([0.9, 0.2])
d1 = np.array([0.8, 0.3])      # points nearly the same way
d2 = np.array([-0.1, 0.9])     # points elsewhere
print("cosine(query, relevant)   =", round(cosine(q, d1), 4))
print("cosine(query, irrelevant) =", round(cosine(q, d2), 4))

cosine(query, relevant)   = 0.9902
cosine(query, irrelevant) = 0.1078


In [37]:
def retrieve(question: str,
             k: int = 4,
             user_roles: Optional[List[str]] = None,
             active_only: bool = False,
             department: Optional[str] = None,
             min_score: float = 0.0) -> pd.DataFrame:
    """Vector search, then the filters that make it safe and correct."""
    qv     = embedder.encode([question])[0]
    scored = []
    for chunk, vec in zip(CHUNKS, INDEX):
        # --- access control BEFORE anything else reaches the model -------------
        if user_roles is not None:
            allowed = set(chunk["access_roles"])
            if "all" not in allowed and not allowed & set(user_roles):
                print(f"Skipping Chunk ID: {chunk['chunk_id']} due to access control")
                continue
        if active_only and chunk["status"] != "active":
            continue
        if department and chunk["department"] != department:
            continue
        score = cosine(qv, vec)
        if score < min_score:          # too weak to be evidence of anything
            continue
        scored.append({**chunk, "score": score})

    df = pd.DataFrame(scored)
    if df.empty:
        return df
    return df.sort_values("score", ascending=False).head(k).reset_index(drop=True)

QUESTION = "How many days do I have to cancel an Enterprise Annual plan?"

print("UNFILTERED — note what comes back at rank 1 and rank 2:")
display(retrieve(QUESTION, k=4)[["chunk_id", "version", "status", "score", "text"]])

UNFILTERED — note what comes back at rank 1 and rank 2:


,chunk_id,version,status,score,text
0,POL-CANCEL-2024#0,2024-03,superseded,0.855588,Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...
1,POL-CANCEL-2026#0,2026-09,active,0.845764,Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...
2,POL-CANCEL-2024#1,2024-03,superseded,0.808218,Enterprise Cancellation Policy (2024-03): must be submitted by an account administrato...
3,POL-CANCEL-2026#1,2026-09,active,0.769998,Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...


**Look at that result before moving on.** Both cancellation policies come back — and the
**superseded 2024 policy outranks the current one**. The two documents are nearly identical
text, so their scores are nearly identical, and the dead one happens to win.

A vector search has no idea one of them is *dead*. It is not a retrieval bug; it is a
**missing filter**. This is the single most common source of confidently-wrong RAG answers
in real deployments.

In [38]:
print("FILTERED to active versions only:")
display(retrieve(QUESTION, k=3, active_only=True)[["chunk_id", "version", "status", "score", "text"]])

print("\nACCESS CONTROL — the same question about salary bands, asked by two different users:")
SALARY_Q = "What are the compensation bands for FY26?"

print("\nROLE 'SUPPORT' VIEW:")
support_view = retrieve(SALARY_Q, k=30, user_roles=["support"], min_score=0.05)

print("\nROLE 'FINANCE' VIEW:")
finance_view = retrieve(SALARY_Q, k=30, user_roles=["finance"], min_score=0.05)

def sees_salary(df):
    return "FIN-SALARY#0" in (df["chunk_id"].tolist() if not df.empty else [])

print(f"\nrole 'support' -> can see the salary document: {sees_salary(support_view)}")

display(support_view[["chunk_id", "department", "score"]] if not support_view.empty
        else "nothing passes the ACL + score filter")

print(f"role 'finance' -> can see the salary document: {sees_salary(finance_view)}")

display(finance_view[["chunk_id", "department", "score"]])

print("\nFiltering happens during retrieval, never after generation.")
print("If an unauthorised chunk reaches the prompt, it has already leaked.")

FILTERED to active versions only:


,chunk_id,version,status,score,text
0,POL-CANCEL-2026#0,2026-09,active,0.845764,Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...
1,POL-CANCEL-2026#1,2026-09,active,0.769998,Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...
2,POL-PROVISION#0,2026-04,active,0.738200,Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...



ACCESS CONTROL — the same question about salary bands, asked by two different users:

ROLE 'SUPPORT' VIEW:
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control

ROLE 'FINANCE' VIEW:
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control

role 'support' -> can see the salary document: False


,chunk_id,department,score
0,POL-CANCEL-2026#1,legal,0.623385
1,SUP-HOURS#1,support,0.620779
2,POL-PROVISION#0,legal,0.607499
3,SUP-HOURS#0,support,0.592160
4,POL-CANCEL-2024#1,legal,0.577657
5,MKT-BLOG#0,marketing,0.566226
6,POL-CANCEL-2026#0,legal,0.556651
7,POL-CANCEL-2024#0,legal,0.512902


role 'finance' -> can see the salary document: True


,chunk_id,department,score
0,FIN-SALARY#0,finance,0.818596
1,POL-CANCEL-2026#1,legal,0.623385
2,SUP-HOURS#1,support,0.620779
3,POL-PROVISION#0,legal,0.607499
4,SUP-HOURS#0,support,0.592160
5,POL-CANCEL-2024#1,legal,0.577657
6,MKT-BLOG#0,marketing,0.566226
7,POL-CANCEL-2026#0,legal,0.556651
8,POL-CANCEL-2024#0,legal,0.512902



Filtering happens during retrieval, never after generation.
If an unauthorised chunk reaches the prompt, it has already leaked.


In [39]:
pd.DataFrame(CHUNKS)

,chunk_id,doc_id,title,version,status,department,access_roles,text
0,POL-CANCEL-2024#0,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...
1,POL-CANCEL-2024#1,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): must be submitted by an account administrato...
2,POL-CANCEL-2026#0,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...
3,POL-CANCEL-2026#1,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...
4,POL-PROVISION#0,POL-PROVISION,Enterprise Provisioning Rules,2026-04,active,legal,[all],Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...
5,SUP-HOURS#0,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): Enterprise customers receive 24x7 support through chat...
6,SUP-HOURS#1,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): 6pm IST on business days.
7,FIN-SALARY#0,FIN-SALARY,Compensation Bands FY26,2026-04,active,finance,[finance],Compensation Bands FY26 (2026-04): Compensation bands for FY26 are reviewed each April...
8,ENG-RUNBOOK#0,ENG-RUNBOOK,Search Latency Runbook,2026-07,active,engineering,[engineering],"Search Latency Runbook (2026-07): If p95 search latency exceeds 800ms, check the vecto..."
9,ENG-RUNBOOK#1,ENG-RUNBOOK,Search Latency Runbook,2026-07,active,engineering,[engineering],"Search Latency Runbook (2026-07): saturation, then the reranker queue depth."


In [40]:
support_view

,chunk_id,doc_id,title,version,status,department,access_roles,text,score
0,POL-CANCEL-2026#1,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...,0.623385
1,SUP-HOURS#1,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): 6pm IST on business days.,0.620779
2,POL-PROVISION#0,POL-PROVISION,Enterprise Provisioning Rules,2026-04,active,legal,[all],Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...,0.607499
3,SUP-HOURS#0,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): Enterprise customers receive 24x7 support through chat...,0.592160
4,POL-CANCEL-2024#1,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): must be submitted by an account administrato...,0.577657
5,MKT-BLOG#0,MKT-BLOG,Why We Rebranded,2025-11,active,marketing,[all],Why We Rebranded (2025-11): Our rebrand focused on clarity and a warmer colour palette...,0.566226
6,POL-CANCEL-2026#0,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...,0.556651
7,POL-CANCEL-2024#0,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...,0.512902


In [41]:
finance_view

,chunk_id,doc_id,title,version,status,department,access_roles,text,score
0,FIN-SALARY#0,FIN-SALARY,Compensation Bands FY26,2026-04,active,finance,[finance],Compensation Bands FY26 (2026-04): Compensation bands for FY26 are reviewed each April...,0.818596
1,POL-CANCEL-2026#1,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): must be submitted by an account administrato...,0.623385
2,SUP-HOURS#1,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): 6pm IST on business days.,0.620779
3,POL-PROVISION#0,POL-PROVISION,Enterprise Provisioning Rules,2026-04,active,legal,[all],Enterprise Provisioning Rules (2026-04): Cancellation eligibility ends once enterprise...,0.607499
4,SUP-HOURS#0,SUP-HOURS,Support Availability,2026-01,active,support,[all],Support Availability (2026-01): Enterprise customers receive 24x7 support through chat...,0.592160
5,POL-CANCEL-2024#1,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): must be submitted by an account administrato...,0.577657
6,MKT-BLOG#0,MKT-BLOG,Why We Rebranded,2025-11,active,marketing,[all],Why We Rebranded (2025-11): Our rebrand focused on clarity and a warmer colour palette...,0.566226
7,POL-CANCEL-2026#0,POL-CANCEL-2026,Enterprise Cancellation Policy,2026-09,active,legal,[all],Enterprise Cancellation Policy (2026-09): Enterprise Annual plans may be cancelled wit...,0.556651
8,POL-CANCEL-2024#0,POL-CANCEL-2024,Enterprise Cancellation Policy,2024-03,superseded,legal,[all],Enterprise Cancellation Policy (2024-03): Enterprise Annual plans may be cancelled wit...,0.512902


### Measuring retrieval

You cannot improve what you do not measure. The two starter metrics:

- **Recall@k** — of the chunks that *should* have been found, how many were in the top k?
- **Hit rate / Success@k** — did at least one correct chunk appear in the top k?

Recall answers "did we find the evidence"; it says nothing about whether the answer was good.
That comes later, in week 9.

In [42]:
def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    if not relevant_ids:
        return 0.0
    # print(f"set(retrieved_ids[:k])=>{set(retrieved_ids[:k])}")
    # print(f"set(relevant_ids)=>{set(relevant_ids)}")
    # print(f"intersection => {set(retrieved_ids[:k]) & set(relevant_ids)}")
    
    return len(set(retrieved_ids[:k]) & set(relevant_ids)) / len(set(relevant_ids))

def hit_rate_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> int:
    return int(bool(set(retrieved_ids[:k]) & set(relevant_ids)))

GOLDEN_SET = [
    ("How many days do I have to cancel an Enterprise Annual plan?", ["POL-CANCEL-2026#0"]),
    ("When does cancellation eligibility end?",                      ["POL-PROVISION#0"]),
    ("Is support available at night for Enterprise?",                ["SUP-HOURS#0"]),
    ("What should I check when search is slow?",                     ["ENG-RUNBOOK#0"]),
]

rows = []
for question, relevant in GOLDEN_SET:
    got = retrieve(question, k=3, active_only=True, user_roles=["all", "engineering"])
    ids = got["chunk_id"].tolist() if not got.empty else []
    rows.append({"question": question[:52] + "...",
                 "top-1": ids[0] if ids else "-",
                 "recall@3": recall_at_k(ids, relevant, 3),
                 "hit@3": hit_rate_at_k(ids, relevant, 3)})

evaluation = pd.DataFrame(rows)
display(evaluation)
print(f"mean recall@3 = {evaluation['recall@3'].mean():.2f}   "
      f"hit rate@3 = {evaluation['hit@3'].mean():.2f}")

Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control


,question,top-1,recall@3,hit@3
0,How many days do I have to cancel an Enterprise Annu...,POL-CANCEL-2026#0,1.0,1
1,When does cancellation eligibility end?...,POL-PROVISION#0,1.0,1
2,Is support available at night for Enterprise?...,SUP-HOURS#0,1.0,1
3,What should I check when search is slow?...,ENG-RUNBOOK#0,1.0,1


mean recall@3 = 1.00   hit rate@3 = 1.00


---
# 13. Generation pipeline

Retrieval found the evidence. Generation has to use it **and nothing else**.

A production prompt has six parts, and skipping any of them shows up later as a bug:

1. **system policy** — the rules of engagement
2. **the question**
3. **the evidence**, each block with a stable id
4. **an output contract** — what shape the answer must take
5. **an abstention rule** — what to do with insufficient evidence
6. **a conflict rule** — what to do when sources disagree

Below, `generate()` calls EURI if you configured a key. Otherwise it runs a small scripted
generator so the pipeline still completes and you can read the prompt it would have sent.

In [43]:
def generate(prompt: str, temperature: float = 0.0) -> str:
    """Real LLM if configured; otherwise a transparent scripted stand-in."""
    if LIVE_LLM:
        client = OpenAI(api_key=EURI_API_KEY, base_url=EURI_BASE_URL)
        r = client.chat.completions.create(
            model=EURI_CHAT_MODEL,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content

    # --- scripted stand-in -------------------------------------------------
    # Pulls the first sentence of [S1] and cites it. Not intelligent; just honest.
    # anchor to the start of a line so we match the EVIDENCE block,
    # not the "e.g. [S1]" that appears inside the rules.
    m = re.search(r"^\[S1\]\s*(.+)$", prompt, re.M)
    if not m:
        return "The available sources do not answer this."
    claim = m.group(1).split(". ")[0].strip().rstrip(".")
    return f"{claim} [S1].\n\n(scripted stand-in — set EURI_API_KEY for a real answer)"

def answer_question(question: str,
                    k: int = 3,
                    user_roles: Optional[List[str]] = None,
                    active_only: bool = True,
                    min_score: float = 0.0) -> Dict[str, Any]:
    hits = retrieve(question, k=k, user_roles=user_roles,
                    active_only=active_only, min_score=min_score)
    if hits.empty:
        return {"question": question, "answer": "The available sources do not answer this.",
                "evidence": pd.DataFrame(), "prompt": None}

    blocks, evidence = [], []
    for i, row in hits.iterrows():
        sid = f"S{i + 1}"
        blocks.append(f'[{sid}] {row["text"]}')
        evidence.append({"source_id": sid, "chunk_id": row["chunk_id"],
                         "doc_id": row["doc_id"], "version": row["version"],
                         "score": round(row["score"], 4)})

    prompt = build_prompt(question, blocks)
    return {"question": question, "answer": generate(prompt),
            "evidence": pd.DataFrame(evidence), "prompt": prompt}

result = answer_question(QUESTION, user_roles=["all"])
print("QUESTION:", result["question"])
print("\nANSWER:\n" + result["answer"])
print("\nEVIDENCE USED:")
display(result["evidence"])

Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
QUESTION: How many days do I have to cancel an Enterprise Annual plan?

ANSWER:
You have 14 calendar days to cancel an Enterprise Annual plan from the initial purchase date, provided that the cancellation request is submitted by an account administrator and the provisioning has not been completed. Cancellation eligibility ends once enterprise provisioning has been completed, even if the 14-day window has not expired [S1][S2][S3].

EVIDENCE USED:


,source_id,chunk_id,doc_id,version,score
0,S1,POL-CANCEL-2026#0,POL-CANCEL-2026,2026-09,0.8458
1,S2,POL-CANCEL-2026#1,POL-CANCEL-2026,2026-09,0.7700
2,S3,POL-PROVISION#0,POL-PROVISION,2026-04,0.7382


In [44]:
print(prompt)

You are a grounded enterprise knowledge assistant.

Rules:
1. Use ONLY the evidence below for company-specific claims.
2. If the evidence is insufficient, say exactly: "The available sources do not answer this."
3. If sources conflict, state the conflict and cite both.
4. Cite every material factual claim with its source id, e.g. [S1].
5. Do not add information from your own training data.

EVIDENCE
[S1] Invoices are issued on the first business day of each month.

QUESTION
how long do I have to cancel?

ANSWER (with citations):


---
# 14. Grounding and citations

Grounding is not one check. It happens at four points, and a system can pass three and fail the fourth:

| Layer | The question it answers |
|---|---|
| Retrieval grounding | Did we *find* the right evidence? |
| Context grounding | Did we actually *send* it to the model? |
| Claim grounding | Is every claim in the answer *supported* by what we sent? |
| Citation grounding | Does each citation point at evidence that *actually supports* that claim? |

A citation is not decoration. It is **data lineage** — enough information to reconstruct
exactly which version of which document produced a claim, months later, in an audit.

In [45]:
def citation_record(evidence_row: Dict[str, Any], corpus: List[Dict[str, Any]]) -> Dict[str, Any]:
    doc = next(d for d in corpus if d["doc_id"] == evidence_row["doc_id"])
    return {
        "source_id":    evidence_row["source_id"],
        "document_id":  doc["doc_id"],
        "chunk_id":     evidence_row["chunk_id"],
        "title":        doc["title"],
        "version":      doc["version"],
        "status":       doc["status"],
        "department":   doc["department"],
        "retrieved_score": evidence_row["score"],
    }

import json
lineage = [citation_record(r, CORPUS) for r in result["evidence"].to_dict("records")]
print(json.dumps(lineage, indent=2))

[
  {
    "source_id": "S1",
    "document_id": "POL-CANCEL-2026",
    "chunk_id": "POL-CANCEL-2026#0",
    "title": "Enterprise Cancellation Policy",
    "version": "2026-09",
    "status": "active",
    "department": "legal",
    "retrieved_score": 0.8458
  },
  {
    "source_id": "S2",
    "document_id": "POL-CANCEL-2026",
    "chunk_id": "POL-CANCEL-2026#1",
    "title": "Enterprise Cancellation Policy",
    "version": "2026-09",
    "status": "active",
    "department": "legal",
    "retrieved_score": 0.77
  },
  {
    "source_id": "S3",
    "document_id": "POL-PROVISION",
    "chunk_id": "POL-PROVISION#0",
    "title": "Enterprise Provisioning Rules",
    "version": "2026-04",
    "status": "active",
    "department": "legal",
    "retrieved_score": 0.7382
  }
]


In [46]:
result["evidence"]

,source_id,chunk_id,doc_id,version,score
0,S1,POL-CANCEL-2026#0,POL-CANCEL-2026,2026-09,0.8458
1,S2,POL-CANCEL-2026#1,POL-CANCEL-2026,2026-09,0.7700
2,S3,POL-PROVISION#0,POL-PROVISION,2026-04,0.7382


In [47]:
CORPUS

[{'doc_id': 'POL-CANCEL-2024',
  'title': 'Enterprise Cancellation Policy',
  'version': '2024-03',
  'status': 'superseded',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Annual plans may be cancelled within 30 calendar days of initial purchase. A cancellation request must be submitted by an account administrator. Refunds are processed within two billing cycles.'},
 {'doc_id': 'POL-CANCEL-2026',
  'title': 'Enterprise Cancellation Policy',
  'version': '2026-09',
  'status': 'active',
  'department': 'legal',
  'access_roles': ['all'],
  'text': 'Enterprise Annual plans may be cancelled within 14 calendar days of initial purchase. A cancellation request must be submitted by an account administrator. Refunds are processed within one billing cycle. This version supersedes the 2024 policy.'},
 {'doc_id': 'POL-PROVISION',
  'title': 'Enterprise Provisioning Rules',
  'version': '2026-04',
  'status': 'active',
  'department': 'legal',
  'access_roles': ['all'],

In [48]:
def validate_citations(answer: str, evidence: pd.DataFrame) -> Dict[str, Any]:
    """Cheap, mechanical checks that catch a surprising share of real failures."""
    cited  = set(re.findall(r"\[(S\d+)\]", answer))
    valid  = set(evidence["source_id"]) if not evidence.empty else set()
    unsupported = cited - valid
    # any sentence making a claim but citing nothing
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", answer) if s.strip()]
    uncited   = [s for s in sentences
                 if not re.search(r"\[S\d+\]", s)
                 and not s.lower().startswith(("the available sources", "(scripted"))]
    return {
        "citations_found":        sorted(cited),
        "invalid_citations":      sorted(unsupported),
        "sentences_without_cite": uncited,
        "passes": not unsupported and not uncited,
    }

check = validate_citations(result["answer"], result["evidence"])
for key, value in check.items():
    print(f"{key:24}: {value}")

print("\nNow the same check on a deliberately bad answer:")
bad = "Enterprise plans can be cancelled within 30 days [S7]. A fee of 5% applies."
display(pd.Series(validate_citations(bad, result["evidence"])))

citations_found         : ['S1', 'S2', 'S3']
invalid_citations       : []
sentences_without_cite  : ['You have 14 calendar days to cancel an Enterprise Annual plan from the initial purchase date, provided that the cancellation request is submitted by an account administrator and the provisioning has not been completed.']
passes                  : False

Now the same check on a deliberately bad answer:


citations_found                             [S7]
invalid_citations                           [S7]
sentences_without_cite    [A fee of 5% applies.]
passes                                     False
dtype: object

In [49]:
result["answer"]

'You have 14 calendar days to cancel an Enterprise Annual plan from the initial purchase date, provided that the cancellation request is submitted by an account administrator and the provisioning has not been completed. Cancellation eligibility ends once enterprise provisioning has been completed, even if the 14-day window has not expired [S1][S2][S3].'

In [50]:
re.split(r"(?<=[.!?])\s+", result["answer"])

['You have 14 calendar days to cancel an Enterprise Annual plan from the initial purchase date, provided that the cancellation request is submitted by an account administrator and the provisioning has not been completed.',
 'Cancellation eligibility ends once enterprise provisioning has been completed, even if the 14-day window has not expired [S1][S2][S3].']

Two failures caught with regular expressions and no model at all: a citation to a source
that was never retrieved, and a factual claim with no citation behind it. Put this in CI.

> **A detail worth stealing:** the citation goes *inside* the sentence it supports —
> `...within 14 calendar days [S1].` — not after the full stop. Put it after, and any
> per-sentence attribution check sees an uncited claim followed by a stray marker.
> Say so explicitly in your system prompt.

---
# 15. Enterprise use cases

Every one of these is "a RAG chatbot". None of them has the same architecture — the
**hard constraint** differs, and that constraint decides your design.

In [51]:
use_cases = pd.DataFrame([
 ["Internal knowledge assistant", "Policies, SOPs, wikis, engineering docs",
  "Per-department access control", "Wrong-department leakage"],
 ["Customer-support copilot", "Product docs, known issues, account context",
  "Latency and tenant isolation", "One customer sees another's data"],
 ["Legal / contract assistant", "Clauses, definitions, obligations",
  "Exact provenance and versioning", "Citing a superseded clause"],
 ["Engineering incident assistant", "Runbooks, postmortems, ownership",
  "Freshness during an incident", "Following a runbook that changed"],
 ["Sales / proposal assistant", "Approved claims, pricing rules, case studies",
  "No invented commitments", "Promising a feature that does not exist"],
 ["Analytics assistant", "Semantic search plus SQL over live tables",
  "Query authorisation, row-level security", "Unbounded query on production"],
], columns=["Use case", "Knowledge sources", "The hard constraint", "The failure that gets you fired"])

display(use_cases)
print('"RAG chatbot" is not an architecture. The constraint column is where the design lives.')

,Use case,Knowledge sources,The hard constraint,The failure that gets you fired
0,Internal knowledge assistant,"Policies, SOPs, wikis, engineering docs",Per-department access control,Wrong-department leakage
1,Customer-support copilot,"Product docs, known issues, account context",Latency and tenant isolation,One customer sees another's data
2,Legal / contract assistant,"Clauses, definitions, obligations",Exact provenance and versioning,Citing a superseded clause
3,Engineering incident assistant,"Runbooks, postmortems, ownership",Freshness during an incident,Following a runbook that changed
4,Sales / proposal assistant,"Approved claims, pricing rules, case studies",No invented commitments,Promising a feature that does not exist
5,Analytics assistant,Semantic search plus SQL over live tables,"Query authorisation, row-level security",Unbounded query on production


"RAG chatbot" is not an architecture. The constraint column is where the design lives.


---
# 16. Putting it together

One question, the full path: retrieve → filter → build context → generate → cite → validate.
Run it for several questions, including one the corpus **cannot** answer.

In [52]:
MIN_EVIDENCE_SCORE = 0.05   # below this, a "match" is noise, not evidence

def rag(question: str, user_roles: List[str] = ["all"], k: int = 3) -> Dict[str, Any]:
    out   = answer_question(question, k=k, user_roles=user_roles,
                            active_only=True, min_score=MIN_EVIDENCE_SCORE)
    out["citation_check"] = validate_citations(out["answer"], out["evidence"])
    return out

questions = [
    ("How many days do I have to cancel an Enterprise Annual plan?", ["all"]),
    ("Is Enterprise support available at night?",                     ["all"]),
    ("What are the FY26 compensation bands?",                         ["support"]),   # not authorised
    ("How do I reset my VPN password?",                               ["all"]),       # not in corpus
]

for q, roles in questions:
    out = rag(q, user_roles=roles)
    print("=" * 100)
    print("Q:", q, f"   (roles={roles})")
    print("A:", out["answer"].split("\n")[0])
    if not out["evidence"].empty:
        print("   sources:", ", ".join(out["evidence"]["chunk_id"]))
    else:
        print("   sources: none retrieved")
    print("   citations valid:", out["citation_check"]["passes"])

Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Q: How many days do I have to cancel an Enterprise Annual plan?    (roles=['all'])
A: You have 14 calendar days to cancel an Enterprise Annual plan from the initial purchase date. However, cancellation eligibility ends once enterprise provisioning has been completed, even if the 14-day window has not expired. The cancellation request must be submitted by an account administrator. Refunds are processed within one billing cycle [S1, S2, S3].
   sources: POL-CANCEL-2026#0, POL-CANCEL-2026#1, POL-PROVISION#0
   citations valid: False
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Q: Is Enterprise support available at night?    (roles=['all'])
A: Yes, Enterprise support is available at night. Enterprise customers r

Look at the last two carefully.

- The **compensation** question returns nothing, because ACL filtering removed the only
  relevant chunk *before* retrieval scored it. That is correct behaviour, not a bug.
- The **VPN** question has no answer in the corpus at all. Every chunk scores essentially zero,
  `MIN_EVIDENCE_SCORE` drops them, and the system abstains instead of answering from the
  nearest-looking chunk. Without that threshold it would have cheerfully quoted the
  cancellation policy.

Retrieval that returns nothing is often the right answer. Systems that cannot say
"I don't know" are the ones that end up in incident reviews.

In [53]:
# --- self-test: run this to confirm the notebook is behaving -----------------
def _self_test():
    checks = []

    c = chunk_text("a b c d e f g h", size=4, overlap=1)
    checks.append(("chunking advances and overlaps", len(c) == 3 and c[0] == "a b c d"))

    try:
        chunk_text("a b c", size=3, overlap=3); ok = False
    except ValueError:
        ok = True
    checks.append(("chunker rejects overlap >= size", ok))

    checks.append(("cosine identity == 1.0", abs(cosine([1, 0], [1, 0]) - 1.0) < 1e-9))
    checks.append(("cosine orthogonal == 0.0", abs(cosine([1, 0], [0, 1])) < 1e-9))

    unf = retrieve(QUESTION, k=6)
    checks.append(("unfiltered retrieval surfaces the superseded policy",
                   "POL-CANCEL-2024#0" in unf["chunk_id"].tolist()))

    act = retrieve(QUESTION, k=6, active_only=True)
    checks.append(("active_only filter removes it",
                   "POL-CANCEL-2024#0" not in act["chunk_id"].tolist()))

    denied  = retrieve("compensation bands FY26", k=5, user_roles=["support"])
    allowed = retrieve("compensation bands FY26", k=5, user_roles=["finance"])
    checks.append(("ACL hides finance doc from support",
                   "FIN-SALARY#0" not in (denied["chunk_id"].tolist() if not denied.empty else [])))
    checks.append(("ACL shows finance doc to finance",
                   "FIN-SALARY#0" in allowed["chunk_id"].tolist()))

    checks.append(("recall@k maths", recall_at_k(["a", "b", "c"], ["b", "z"], 3) == 0.5))
    checks.append(("hit@k maths",    hit_rate_at_k(["a", "b"], ["b"], 2) == 1))

    # the stale policy actually outranks the current one when unfiltered -- the
    # whole point of the filtering section, so assert it rather than hope
    ids_unf = unf["chunk_id"].tolist()
    checks.append(("stale policy outranks current when unfiltered",
                   ids_unf.index("POL-CANCEL-2024#0") < ids_unf.index("POL-CANCEL-2026#0")))

    r = rag("How many days do I have to cancel an Enterprise Annual plan?")
    checks.append(("answer cites the current policy",
                   "POL-CANCEL-2026#0" in r["evidence"]["chunk_id"].tolist()))
    # a generator that silently emits an empty claim is the bug this catches
    claim = r["answer"].split("\n")[0]
    checks.append(("answer contains a real claim, not an empty one", len(claim.strip(" .[]S1")) > 20))
    checks.append(("answer states the CURRENT window (14 days)", "14 calendar days" in r["answer"]))
    checks.append(("answer does not state the stale window (30 days)", "30 calendar days" not in r["answer"]))
    checks.append(("citations on the answer validate", r["citation_check"]["passes"]))
    checks.append(("citation validator rejects a fabricated source",
                   validate_citations("claim [S99].", r["evidence"])["invalid_citations"] == ["S99"]))

    # out-of-corpus question must abstain, not answer from the nearest chunk
    vpn = rag("How do I reset my VPN password?")
    checks.append(("out-of-corpus question retrieves nothing", vpn["evidence"].empty))
    checks.append(("out-of-corpus question abstains",
                   vpn["answer"] == "The available sources do not answer this."))

    # unauthorised question must also abstain
    sal = rag("What are the FY26 compensation bands?", user_roles=["support"])
    checks.append(("unauthorised question abstains", sal["evidence"].empty))

    print(f"{'CHECK':<52} RESULT")
    print("-" * 62)
    for name, ok in checks:
        print(f"{name:<52} {'pass' if ok else 'FAIL'}")
    failed = [n for n, ok in checks if not ok]
    print("-" * 62)
    print("ALL CHECKS PASSED" if not failed else f"{len(failed)} FAILED: {failed}")
    return not failed

_self_test()

Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
Skipping Chunk ID: FIN-SALARY#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#0 due to access control
Skipping Chunk ID: ENG-RUNBOOK#1 due to access control
CHECK                                                RESULT
--------------------------------------------------------------
chunking advances and overlaps                       pass
chunker rejects overlap >= size                      

False